In [23]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from binance.client import Client
import os 
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model = 'llama-3.3-70b-versatile')

In [4]:
# Your API credentials
api_key = os.getenv("BINANCE_API_KEY")
api_secret = os.getenv("BINANCE_SECRET_KEY")

# Initialize client
client = Client(api_key, api_secret)

In [21]:
@tool
def get_price(symbol: str):
    """Fetch latest price of the given symbol/coin """
    # If symbol doesn't end with USDT, append it
    if not symbol.endswith(('USDT', 'BUSD', 'USDC')):
        symbol = symbol + 'USDT'
    
    try:
        ticker = client.get_symbol_ticker(symbol=symbol)
        return float(ticker['price'])
    except Exception as e:
        return f"Error: Symbol '{symbol}' not found or invalid. {str(e)}"

In [22]:
tools = [get_price]
llm_with_tools = llm.bind_tools(tools)

In [7]:
class ToolState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [24]:
def chat_node(state: ToolState):
    """LLM model taht can answer or call a tool """
    message = state['messages']

    system_msg = SystemMessage(content="You are a helpful assistant that can answer questions and call tools when needed.")
    respone = llm_with_tools.invoke([system_msg] + message)

    return {'messages': [respone]}

tool_node = ToolNode(tools)

In [25]:
graph = StateGraph(ToolState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

workflow= graph.compile()

In [30]:
response  = workflow.invoke({'messages': [HumanMessage(content='briefly tell me about MYX coin , what move it can take in future , whether to long/short ')]})

In [28]:
response  = workflow.invoke({'messages': [HumanMessage(content='last date of your knowledge update ')]})

In [11]:
print(response['messages'][-1].content)

Based on the current price of BTC ($63502.76), it's difficult to predict with certainty the future movements of the coin. However, I can provide some general insights.

BTC, or Bitcoin, is a highly volatile cryptocurrency that has experienced significant price fluctuations in the past. Its price can be influenced by a variety of factors, including:

1. Market sentiment: Investor attitudes and emotions can drive price movements.
2. Global economic trends: Economic downturns or upswings can impact BTC's price.
3. Regulatory developments: Changes in government regulations or policies can affect the cryptocurrency market.
4. Technological advancements: Improvements in blockchain technology or the development of new cryptocurrencies can impact BTC's price.

As for whether to long or short BTC, it ultimately depends on your personal investment strategy and risk tolerance. Some potential future moves that BTC could take include:

1. Continued growth: If investor sentiment remains positive and